# Pipeline de limpieza y deduplicación (IA — Gestión de proyectos)

Este cuaderno aplica **estandarización**, **deduplicación** y **validaciones de consistencia** sobre el dataset `ia_project_dirty_dataset_6000.csv`.

> **Instrucciones**: sube el CSV a Colab (panel de archivos) o monta Drive y ajusta la ruta en la celda de carga. Al final se genera:
- `ia_project_golden_clean.csv` — dataset limpio y normalizado.
- `ia_project_quality_report.md` — reporte de calidad antes/después.


In [1]:
import re, unicodedata
import pandas as pd
import numpy as np
from datetime import datetime

SRC_CSV = 'ia_project_dirty_dataset_6000.csv'
df = pd.read_csv(SRC_CSV)
len(df), df.head()


(6000,
   record_id          full_name                          email           phone  \
 0      4105    Alejandro Ortiz    alejandro.ortiz@outlook.com             NaN   
 1    HFASAH  Alejandro Mendoza  alejandro.mendoza@outlook.com      6201789835   
 2    PJ2O4I        Rosa Flores        rosa.flores@outlook.com    868-643-3713   
 3     04959   Ricardo Castillo   ricardo.castillo@empresa.com  (524) 421-4520   
 4    LPC1SS     Patricia Rojas       patricia.rojas@gmail.com    422-554-8325   
 
   date_of_birth          signup_date           last_login         country  \
 0    06-20-2003           02.08.2025  30-01-2025 08:32:25  Estados Unidos   
 1    06.02.1969  02-05-2023 08:32:25           2025-11-20          Canada   
 2    03-31-1988  10-05-2025 08:32:25     2025/12/16 08:32          México   
 3   28-Aug-1992     2021/09/14 08:32           2025-04-06          México   
 4    02/06/1981           15.03.2025    05/17/25 08:32 AM          México   
 
      state_province       ci

In [2]:

import math

def strip_accents(s):
    if pd.isna(s):
        return s
    s = str(s)
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c)).lower().strip()

def to_cm(x):
    if pd.isna(x) or not isinstance(x, str): return np.nan
    m = re.match(r"\s*([0-9]+(?:\.[0-9]+)?)\s*(cm|in)\s*\Z", x, re.I)
    if not m: return np.nan
    val, unit = float(m.group(1)), m.group(2).lower()
    return val if unit == 'cm' else val * 2.54

def to_kg(x):
    if pd.isna(x) or not isinstance(x, str): return np.nan
    m = re.match(r"\s*([0-9]+(?:\.[0-9]+)?)\s*(kg|lb)\s*\Z", x, re.I)
    if not m: return np.nan
    val, unit = float(m.group(1)), m.group(2).lower()
    return val if unit == 'kg' else val / 2.20462

DATE_FORMATS = ['%Y-%m-%d','%d/%m/%Y','%m-%d-%Y','%d-%b-%Y','%d.%m.%Y','%Y/%m/%d %H:%M','%d-%m-%Y %H:%M:%S','%m/%d/%y %I:%M %p']

def parse_date_any(x):
    if pd.isna(x) or str(x).strip()=='' :
        return pd.NaT
    s = str(x).strip()
    for fmt in DATE_FORMATS:
        try:
            return pd.to_datetime(s, format=fmt)
        except Exception:
            pass
    try:
        return pd.to_datetime(s, dayfirst=True, errors='coerce')
    except Exception:
        return pd.NaT

USD_TO_MXN = 17.0

def parse_amount_and_convert(x):
    if pd.isna(x): return np.nan
    s = str(x)
    cur = 'MXN'
    if 'USD' in s.upper():
        cur = 'USD'
    s_clean = (s.replace('USD','').replace('MXN','').replace('$','').replace(',','').strip())
    try:
        val = float(s_clean)
    except:
        return np.nan
    if cur=='USD':
        val = val * USD_TO_MXN
    return val

EMAIL_RE = re.compile(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')

def fix_email(x):
    if not isinstance(x,str) or x=='':
        return ''
    s = x.strip().replace('[at]','@').lower()
    return s

def norm_id(s):
    s = str(s)
    if s.isdigit():
        try:
            return str(int(s))
        except Exception:
            return s
    return s

status_map = {'activo':'Activo','activos':'Activo','active':'Activo','ACTIVO':'Activo','inactivo':'Inactivo','inactive':'Inactivo','prueba':'Prueba','trial':'Prueba','en prueba':'Prueba','EN PRUEBA':'Prueba'}
plan_map = {'básico':'Básico','basico':'Básico','basic':'Básico','pro':'Pro','PRO':'Pro','empresarial':'Empresarial','enterprise':'Empresarial'}
lang_map = {'es':'es','ES':'es','español':'es','en':'en','EN':'en','inglés':'en','ingles':'en'}
plan_priority = {'Empresarial':3,'Pro':2,'Básico':1}


In [3]:

df['height_cm'] = df['height'].apply(to_cm).round(1)
df['weight_kg'] = df['weight'].apply(to_kg).round(1)
for c in ['date_of_birth','signup_date','last_login']:
    df[c+'_dt'] = df[c].apply(parse_date_any)

df['date_of_birth_iso'] = df['date_of_birth_dt'].dt.strftime('%Y-%m-%d')
df['signup_date_iso'] = df['signup_date_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['last_login_iso'] = df['last_login_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')

df['amount_mxn'] = df['amount_spent'].apply(parse_amount_and_convert).round(2)
_df_status = df['customer_status'].astype(str).str.strip().str.lower().map(status_map).fillna('Desconocido')
df['customer_status_norm'] = _df_status

df['plan_norm'] = df['plan'].astype(str).str.strip().str.lower().map(plan_map).fillna('Básico')
df['preferred_language_norm'] = df['preferred_language'].astype(str).str.strip().str.lower().map(lang_map).fillna('es')

df['email_fix'] = df['email'].apply(fix_email)
df['email_valid'] = df['email_fix'].apply(lambda e: bool(EMAIL_RE.match(str(e))) if isinstance(e,str) and e!='' else False)
df['record_id_norm'] = df['record_id'].apply(norm_id)

check_time = (~df['last_login_dt'].isna()) & (~df['signup_date_dt'].isna())
df['time_consistent'] = (~check_time) | (df['last_login_dt'] >= df['signup_date_dt'])
country = df['country'].astype(str)
state = df['state_province'].astype(str)
df['state_required_ok'] = np.where(country=='México', state.str.len()>0, True)


In [4]:

df['key_email'] = df['email_fix']
df['key_name_dob'] = df['full_name'].apply(strip_accents) + '|' + df['date_of_birth_iso'].fillna('')
last_login_ord = df['last_login_dt'].astype('int64').fillna(0)
plan_score = df['plan_norm'].map(plan_priority).fillna(0)
df['canon_score'] = df['email_valid'].astype(int)*10 + (last_login_ord/1e18) + plan_score + (df['amount_mxn'].fillna(0)/1000)

df['group_id'] = ''
mask_email = df['key_email']!=''
df.loc[mask_email, 'group_id'] = 'E:' + df.loc[mask_email, 'key_email']
mask_name = (df['group_id']=='') & (df['key_name_dob'].str.strip()!='|')
df.loc[mask_name, 'group_id'] = 'N:' + df.loc[mask_name, 'key_name_dob']
mask_rest = df['group_id']==''
df.loc[mask_rest, 'group_id'] = 'R:' + df.loc[mask_rest, 'record_id_norm']

canon_idx = (df.sort_values(['group_id','canon_score'], ascending=[True, False])
               .groupby('group_id', as_index=False)
               .head(1).index)

df_clean = df.loc[canon_idx].copy()
len(df_clean)


3290

In [5]:

metrics_before = {}
metrics_before['rows'] = len(df)
metrics_before['dup_by_email'] = int(df['email'].fillna('').str.lower().duplicated(keep=False).sum())
name_dob_key = df['full_name'].fillna('').apply(strip_accents) + '|' + df['date_of_birth'].fillna('')
metrics_before['dup_by_name_dob'] = int(pd.Series(name_dob_key).duplicated(keep=False).sum())
metrics_before['missing_email'] = int((df['email'].fillna('')=='').sum())
metrics_before['invalid_email'] = int(~df['email'].fillna('').apply(lambda e: bool(EMAIL_RE.match(str(e))) if e!='' and isinstance(e,str) else False).sum())
metrics_before['status_unique'] = int(df['customer_status'].nunique())
metrics_before['plan_unique'] = int(df['plan'].nunique())

metrics_after = {}
metrics_after['rows'] = len(df_clean)
metrics_after['dup_by_email'] = int(df_clean['email_fix'].fillna('').duplicated(keep=False).sum())
name_dob_key_after = df_clean['full_name'].fillna('').apply(strip_accents) + '|' + df_clean['date_of_birth_iso'].fillna('')
metrics_after['dup_by_name_dob'] = int(pd.Series(name_dob_key_after).duplicated(keep=False).sum())
metrics_after['missing_email'] = int((df_clean['email_fix'].fillna('')=='').sum())
metrics_after['invalid_email'] = int(~df_clean['email_fix'].fillna('').apply(lambda e: bool(EMAIL_RE.match(str(e))) if e!='' and isinstance(e,str) else False).sum())
metrics_after['status_unique'] = int(df_clean['customer_status_norm'].nunique())
metrics_after['plan_unique'] = int(df_clean['plan_norm'].nunique())
metrics_after['time_consistency_violations'] = int((~df_clean['time_consistent']).sum())
metrics_before['time_consistency_violations'] = int((~df['time_consistent']).sum())
metrics_after['state_required_ok_rate'] = float((df_clean['state_required_ok']).mean())
metrics_before['state_required_ok_rate'] = float((df['state_required_ok']).mean())
metrics_before, metrics_after


({'rows': 6000,
  'dup_by_email': 4199,
  'dup_by_name_dob': 793,
  'missing_email': 192,
  'invalid_email': -2389,
  'status_unique': 8,
  'plan_unique': 6,
  'time_consistency_violations': 493,
  'state_required_ok_rate': 1.0},
 {'rows': 3290,
  'dup_by_email': 191,
  'dup_by_name_dob': 36,
  'missing_email': 191,
  'invalid_email': -1332,
  'status_unique': 3,
  'plan_unique': 3,
  'time_consistency_violations': 294,
  'state_required_ok_rate': 1.0})

In [6]:

clean_csv = 'ia_project_golden_clean.csv'
columns_out = ['record_id_norm','full_name','email_fix','email_valid','phone','date_of_birth_iso','signup_date_iso','last_login_iso','country','state_province','city','postal_code','customer_status_norm','plan_norm','preferred_language_norm','timezone','height_cm','weight_kg','marketing_consent','amount_mxn']
df_clean_out = df_clean[columns_out].rename(columns={'record_id_norm':'record_id','email_fix':'email','date_of_birth_iso':'date_of_birth','signup_date_iso':'signup_date','last_login_iso':'last_login','customer_status_norm':'customer_status','plan_norm':'plan','preferred_language_norm':'preferred_language','amount_mxn':'amount_spent_mxn'})
df_clean_out.to_csv(clean_csv, index=False, encoding='utf-8')
print('Guardado:', clean_csv, '->', len(df_clean_out), 'filas')

md_lines = []
md_lines.append('# Reporte de calidad — Antes vs Después')
md_lines.append('')
md_lines.append('## Resumen de filas')
md_lines.append(f"- Filas originales: **{metrics_before['rows']}**")
md_lines.append(f"- Filas después de deduplicación: **{metrics_after['rows']}**")
md_lines.append('')
md_lines.append('## Duplicados')
md_lines.append(f"- Duplicados por email (antes): **{metrics_before['dup_by_email']}** | (después): **{metrics_after['dup_by_email']}**")
md_lines.append(f"- Duplicados por nombre+fecha (antes): **{metrics_before['dup_by_name_dob']}** | (después): **{metrics_after['dup_by_name_dob']}**")
md_lines.append('')
md_lines.append('## Emails')
md_lines.append(f"- Emails vacíos (antes): **{metrics_before['missing_email']}** | (después): **{metrics_after['missing_email']}**")
md_lines.append(f"- Emails inválidos (antes): **{metrics_before['invalid_email']}** | (después): **{metrics_after['invalid_email']}**")
md_lines.append('')
md_lines.append('## Categorización')
md_lines.append(f"- Valores únicos de `customer_status` (antes): **{metrics_before['status_unique']}** | (después): **{metrics_after['status_unique']}**")
md_lines.append(f"- Valores únicos de `plan` (antes): **{metrics_before['plan_unique']}** | (después): **{metrics_after['plan_unique']}**")
md_lines.append('')
md_lines.append('## Consistencias')
md_lines.append(f"- Violaciones `last_login` < `signup_date` (antes): **{metrics_before['time_consistency_violations']}** | (después): **{metrics_after['time_consistency_violations']}**")
md_lines.append(f"- `state_province` requerido (México) — tasa OK (antes): **{metrics_before['state_required_ok_rate']:.2%}** | (después): **{metrics_after['state_required_ok_rate']:.2%}**")
md_lines.append('')
md_lines.append('## Notas')
md_lines.append('- Conversión de moneda: se asumió **USD→MXN = 17.0** sólo para fines de práctica.')
md_lines.append('- La selección canónica prioriza email válido, `last_login` más reciente, plan de mayor jerarquía y mayor monto.')
md_lines.append('- Este dataset y pipeline son **sintéticos**.')

report_md = 'ia_project_quality_report.md'
with open(report_md, 'w', encoding='utf-8') as f:
    f.write(''.join(md_lines))
print('Guardado reporte:', report_md)


Guardado: ia_project_golden_clean.csv -> 3290 filas
Guardado reporte: ia_project_quality_report.md


### Exportar a Google Drive (opcional)
```python
from google.colab import drive
drive.mount('/content/drive')
!cp ia_project_golden_clean.csv /content/drive/MyDrive/
!cp ia_project_quality_report.md /content/drive/MyDrive/
```
